**Important Libraries**

In [8]:
from pyspark.sql import SparkSession
import boto3
import os

In [9]:

# Configure Spark to use localhost for local execution
# This avoids networking issues between Spark driver and executor
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

**Cell 2: Create Spark Session with S3 Connector**

In [10]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("S3_Data_Read_Pipeline")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


spark

**Cell 3: Verify Spark and Hadoop versions**

In [11]:
# Verify environment

print("Spark Version:",
      spark.version)


print(
    "Hadoop Version:",
    spark.sparkContext
    ._jvm
    .org.apache.hadoop.util.VersionInfo
    .getVersion()
)

Spark Version: 4.1.1
Hadoop Version: 3.4.2


**Cell 4: Verify AWS credentials using boto3**

In [12]:
# This checks whether AWS credentials are available
# Credentials are picked from AWS CLI configuration
# No secrets are exposed here

sts_client = boto3.client("sts")

identity = sts_client.get_caller_identity()

# print("AWS Account Connected")
# print("Account:", identity["Account"])
# print("User ARN:", identity["Arn"])

**Cell 5: Read CSV from S3 using Spark**

In [13]:
orders = spark.read.csv(
    "s3a://deh-pyspark-challenge-519749210589/data/orders.csv",
    header=True,
    inferSchema=True
)
orders.show(5)

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|
|   O0004|       C004|      P006|2023-01-12|       2|     89.99|           5|Delivered|    Debit Card|  South|
|   O0005|       C005|      P002|2023-01-15|       3|     29.99|           0|Delivered|   Credit Card|   West|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
o

In [14]:
#